# Corticall - Gate 0 v2: frontal-face-present vs face-absent (GO / NO-GO)

Kaggle **GPU T4 + Internet ON**. Recipe: **Run All -> Restart & Clear -> Run All**.

Fixes the AMBIGUOUS Run 1. Pre-registered design (D021 / `notebooks/gate0_v2_stimuli.json`): **FACE** = single frontal face fills the frame; **NONFACE** = no dominant frontal face (people-with-backs/profiles/scenes) - both from the **same** public-domain film *Charade (1963)*, so the only systematic difference is a frontal face (controls film/people/scene/low-level at once). **GO** = right-FFC face>nonface (perm p<=0.025), > V1 and > EBA (specificity), surviving video-only. McLintock landscapes give a PPA place positive-control (reported). ~68 passes, ~7h, HDF5-cached.

## Setup (reused verbatim from `01_setup_test.ipynb`)
### Phase 1 - install TRIBE v2 (+ shadow/numpy guards)

In [ ]:
import sys, subprocess, shutil, importlib
print('Python:', sys.version)
SRC = '/kaggle/working/tribev2_src'
shutil.rmtree('/kaggle/working/tribev2', ignore_errors=True)
shutil.rmtree(SRC, ignore_errors=True)
for _m in [m for m in list(sys.modules) if m == 'tribev2' or m.startswith('tribev2.')]:
    sys.modules.pop(_m, None)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=False)
subprocess.run(f'git clone --depth 1 https://github.com/facebookresearch/tribev2.git {SRC}',
               shell=True, check=False)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', SRC],
                   capture_output=True, text=True)
print('tribev2 install rc =', r.returncode)
if r.returncode != 0:
    print('--- STDERR (tail) ---'); print(r.stderr[-3000:])
    print('>>> If this mentions requires-python / neuralset>=3.12, Kaggle is on Python <3.12 (G016).')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps',
                '--no-cache-dir', '-q', 'numpy==2.2.6'], check=False)
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
print('install step done - inspect above for resolver/version errors')
import numpy as _np
print('numpy:', _np.__version__)
import tribev2
where = getattr(tribev2, '__file__', None) or list(getattr(tribev2, '__path__', []))
print('tribev2 resolves to:', where)
assert where and 'tribev2_src' in str(where), (
    f'SHADOWED: tribev2 resolved to {where}, not {SRC}. Restart (Factory reset) and re-run.')
importlib.import_module('tribev2.demo_utils')
print('OK: tribev2.demo_utils imports.')

### Clone Corticall (tribe-bench) - brings `tribe_tools.roi_stats` etc.

In [ ]:
import os, sys, subprocess, glob
from pathlib import Path
TB_PATH = None
subprocess.run('git clone --depth 1 https://github.com/codesbydevesh/tribe-bench.git /kaggle/working/tribe-bench',
               shell=True, check=False)
if Path('/kaggle/working/tribe-bench/tribe_tools/model.py').is_file():
    TB_PATH = '/kaggle/working/tribe-bench'
if TB_PATH is None:
    for cand in glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*'):
        if Path(cand, 'tribe_tools', 'model.py').is_file():
            TB_PATH = cand; break
if TB_PATH:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', TB_PATH], check=False)
    if TB_PATH not in sys.path:
        sys.path.insert(0, TB_PATH)
    print('tribe-bench found at:', TB_PATH)
else:
    print('tribe-bench NOT available - check the clone error above (repo should be public).')

### Phase 2 - environment

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
else:
    print('NO GPU - enable a T4 accelerator in Kaggle Settings, then re-run.')
import shutil
print('ffmpeg on PATH:', bool(shutil.which('ffmpeg')))
print('uvx on PATH   :', bool(shutil.which('uvx')), '(required for WhisperX ASR)')

### Phase 3 - HuggingFace login (gated LLaMA-3.2)

In [ ]:
import os
hf_ok = False
try:
    from huggingface_hub import login
    token = os.environ.get('HF_TOKEN', '')
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            token = ''
    if token:
        login(token=token); os.environ['HF_TOKEN'] = token; hf_ok = True
        print('HF login OK')
    else:
        print('No HF_TOKEN. Set a Kaggle secret HF_TOKEN. The text (LLaMA) extractor will fail without it.')
except Exception as e:
    print('HF login error:', type(e).__name__, e)

### Phase 4 - import the wrapper

In [ ]:
WRAPPER_OK = False
try:
    from tribe_tools.model import load_model, predict_single, MODALITY_MASKS
    WRAPPER_OK = True
    print('tribe_tools.model imports: OK'); print('MODALITY_MASKS:', MODALITY_MASKS)
except Exception as e:
    print('WRAPPER IMPORT FAILED:', type(e).__name__, e)
for _m in ['tribe_tools.atlas', 'tribe_tools.cache', 'tribe_tools.viz',
           'tribe_tools.roi_stats', 'neurocheck.claims']:
    try:
        __import__(_m); print('  optional import OK:', _m)
    except Exception as e:
        print('  optional import skipped:', _m, '->', type(e).__name__, e)

## Gate 0 v2 prep - download films, cut clips from the frozen manifest, atlas pre-flight

In [ ]:
# ===== GATE 0 v2 - PREP: download 2 PD films, cut clips from the FROZEN manifest =====
import json, subprocess
from pathlib import Path
import numpy as np

CACHE = Path('/kaggle/working/cache'); CACHE.mkdir(parents=True, exist_ok=True)
CLIPS = Path('/kaggle/working/clips'); CLIPS.mkdir(parents=True, exist_ok=True)
M = json.load(open('/kaggle/working/tribe-bench/notebooks/gate0_v2_stimuli.json'))
DUR = M['clip_dur_s']

def fetch(url, out):
    out = Path(out)
    if not out.exists() or out.stat().st_size == 0:
        subprocess.run(['bash', '-lc', f'curl -L --fail -o "{out}" "{url}"'], check=False)
    return out

charade = fetch(M['primary_source']['url'], CACHE / 'charade.mp4')
mcl = fetch(M['confirmatory_scene_source']['url'], CACHE / 'mclintock.mp4')
print('charade:', charade.stat().st_size // 1_000_000, 'MB | mclintock:', mcl.stat().st_size // 1_000_000, 'MB')

def cut(src, start, name):
    out = CLIPS / f'{name}.mp4'
    subprocess.run(['ffmpeg', '-y', '-ss', str(start), '-t', str(DUR), '-i', str(src),
                    '-vf', 'scale=-2:480', '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '23',
                    '-c:a', 'aac', str(out)], check=False, capture_output=True)
    return out

FACE = [cut(charade, s, f'FACE_{i:02d}') for i, s in enumerate(M['face_starts_s'])]
NONFACE = [cut(charade, s, f'NONFACE_{i:02d}') for i, s in enumerate(M['nonface_starts_s'])]
SCENE = [cut(mcl, s, f'SCENE_{i:02d}') for i, s in enumerate(M['confirmatory_scene_source']['scene_starts_s'])]
print('cut:', len(FACE), 'face,', len(NONFACE), 'nonface,', len(SCENE), 'scene(confirm)')

import matplotlib.pyplot as plt
def montage(clips, title, fname):
    n = len(clips); cols = 5; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 2.6 * rows))
    axf = np.atleast_1d(axes).ravel()
    for ax, p in zip(axf, clips):
        th = CLIPS / f'thumb_{p.stem}.png'
        subprocess.run(['ffmpeg', '-y', '-ss', '5', '-i', str(p), '-frames:v', '1', str(th)],
                       check=False, capture_output=True)
        try: ax.imshow(plt.imread(th))
        except Exception: pass
        ax.set_title(p.stem, fontsize=8); ax.axis('off')
    for ax in axf[n:]: ax.axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.savefig(f'/kaggle/working/{fname}'); plt.show()
montage(FACE, 'FACE (frontal face fills frame)', 'm_face.png')
montage(NONFACE, 'NONFACE (no dominant frontal face)', 'm_nonface.png')
print('>>> sanity: top montage = clear single faces; bottom = people-with-backs/profiles/scenes, no close-up face.')

# atlas pre-flight: FFC(face), EBA proxy(body), V1, A1, PPA
from tribe_tools import atlas
from tribe_tools.cache import get_cache
regions = set(atlas.list_regions())
def verts(names, hemi='both'):
    if isinstance(names, str): names = [names]
    keep = [n for n in names if n in regions]
    return np.concatenate([atlas.get_vertices(n, hemi=hemi) for n in keep]) if keep else np.array([], dtype=int)
FFCr = verts('FFC', 'right')
EBA = verts(['LO2', 'LO3', 'V4t', 'FST', 'PH'], 'right')      # body region (replaces the 12-vtx LO2)
V1v = verts('V1'); A1v = verts('A1')
PPA = verts(['PHA1', 'PHA2', 'PHA3', 'VMV1', 'VMV2', 'VMV3'])
for nm, v in [('FFCr', FFCr), ('EBA', EBA), ('V1', V1v), ('A1', A1v), ('PPA', PPA)]:
    print(f'  {nm}: {len(v)} vertices'); assert len(v) > 0, nm
cache = get_cache(CACHE / 'gate0v2')
print('PREP OK. ~68 passes x ~6 min ~= 7h (HDF5-cached, resumes if the session dies).')

## Load the model (reused; uses CACHE from prep)

In [ ]:
import time
tribe = None; load_time = None
if WRAPPER_OK:
    try:
        t0 = time.time()
        tribe = load_model(device='cuda', cache_folder=CACHE)
        load_time = time.time() - t0
        print(f'Model loaded in {load_time:.1f}s')
    except Exception as e:
        import traceback; traceback.print_exc()
        print('MODEL LOAD FAILED:', type(e).__name__, e)
else:
    print('Skipped: wrapper not importable.')

## FULL passes

In [ ]:
# ===== GATE 0 v2 - FULL passes: FACE + NONFACE + SCENE(confirm) =====
from tribe_tools.model import predict_single
def run(v, mask):
    key = f"{v.resolve()}_{'full' if not mask else 'vid'}"
    hit = cache.load(key)
    if hit is not None:
        print('  cache', v.name); return hit
    preds, _ = predict_single(tribe, v, features_to_mask=mask)
    cache.save(key, preds, metadata={'clip': v.name, 'mask': 'full' if not mask else 'vid'})
    print('  ran', v.name, preds.shape); return preds

assert tribe is not None, 'model not loaded'
full = {v.name: run(v, None) for v in FACE + NONFACE + SCENE}
print('FULL done:', len(full))

## VIDEO-ONLY passes (G4 control)

In [ ]:
# ===== GATE 0 v2 - VIDEO-ONLY passes (G4: rules out a speech/audio artifact) =====
vid = {v.name: run(v, ['audio', 'text']) for v in FACE + NONFACE}
print('VIDEO-ONLY done:', len(vid))

## Analysis + pre-registered verdict

In [ ]:
# ===== GATE 0 v2 - analysis + PRE-REGISTERED verdict (D021) =====
import json
from pathlib import Path
import numpy as np
try:
    from tribe_tools.roi_stats import spatial_z, u_statistic, perm_p
except Exception:
    def spatial_z(preds, v):
        g = preds.mean(0) if preds.ndim == 2 else np.asarray(preds); sd = g.std()
        return 0.0 if sd == 0 else float((g[v].mean() - g.mean()) / sd)
    def u_statistic(a, b):
        u = 0.0
        for x in a:
            for y in b: u += 1.0 if x > y else (0.5 if x == y else 0.0)
        return u
    def perm_p(a, b, n_perm=10000, seed=0):
        vals = np.array(list(a) + list(b), float); n = len(a); N = len(vals)
        uo = u_statistic(vals[:n], vals[n:]); rng = np.random.default_rng(seed); ge = 0
        for _ in range(n_perm):
            p = rng.permutation(N)
            if u_statistic(vals[p[:n]], vals[p[n:]]) >= uo - 1e-9: ge += 1
        return (ge + 1) / (n_perm + 1)

def mc_delta_thr(a, b, n_perm=10000, seed=1, pct=95):
    vals = np.array(list(a) + list(b), float); n = len(a); N = len(vals); rng = np.random.default_rng(seed)
    ds = np.array([(lambda p: vals[p[:n]].mean() - vals[p[n:]].mean())(rng.permutation(N)) for _ in range(n_perm)])
    return float(np.percentile(ds, pct))

FN = [p.name for p in FACE]; NN = [p.name for p in NONFACE]; SN = [p.name for p in SCENE]
def zvals(passes, v, names): return [spatial_z(passes[n], v) for n in names]
def report(passes, v, A, B, name):
    a = zvals(passes, v, A); b = zvals(passes, v, B)
    d = float(np.mean(a) - np.mean(b)); U = u_statistic(a, b); p = perm_p(a, b, 10000, 0)
    print(f'{name:12} d={d:+.3f}  U={U:.0f}/{len(a)*len(b)}  p={p:.4f}')
    return dict(name=name, d=d, U=float(U), p=float(p), a=a, b=b)

print('=== FULL: FACE vs NONFACE ===')
FFC = report(full, FFCr, FN, NN, 'FFCr(FFA)'); V1 = report(full, V1v, FN, NN, 'V1')
EB = report(full, EBA, FN, NN, 'EBA(body)'); A1 = report(full, A1v, FN, NN, 'A1')
print('=== VIDEO-ONLY (G4) ==='); FFCv = report(vid, FFCr, FN, NN, 'FFCr vid')
print('=== CONFIRM: SCENE vs FACE in PPA (place control) ==='); PP = report(full, PPA, SN, FN, 'PPA scene>face')

G1 = FFC['p'] <= 0.025
G2 = FFC['d'] > mc_delta_thr(FFC['a'], FFC['b'])
G3 = (FFC['d'] > V1['d']) and (FFC['d'] > EB['d'])
G4 = (FFCv['d'] > 0) and (FFCv['p'] <= 0.05)
if all([G1, G2, G3, G4]):
    verdict = 'GO -> ROADMAP Phase 1'
elif FFC['d'] <= 0 or V1['d'] >= FFC['d'] or EB['d'] >= FFC['d'] or FFCv['d'] <= 0:
    verdict = 'NO-GO -> stop; D017 static-resource fallback'
else:
    verdict = 'AMBIGUOUS -> diagnose (finer curation / stock-video key), do not build Phase 1'

print(f"\nGATES  G1(sig)={G1}  G2(mag)={G2}  G3(spec: FFC>V1 & FFC>EBA)={G3}  G4(video-only)={G4}")
print('VERDICT:', verdict)
print('confirm PPA scene>face: d=%.3f p=%.4f (expect a strong positive place effect)' % (PP['d'], PP['p']))

import matplotlib.pyplot as plt
rows = [FFC, V1, EB, A1]
plt.figure(figsize=(7, 4)); plt.bar([r['name'] for r in rows], [r['d'] for r in rows])
plt.axhline(0, color='k', lw=.8); plt.ylabel('Δ spatial-z (face - nonface)'); plt.title('Gate 0 v2: ' + verdict.split(' ->')[0])
plt.xticks(rotation=15); plt.tight_layout(); plt.savefig('/kaggle/working/gate0v2_contrast.png', dpi=120); plt.show()
print('top-k FACE ROIs:', atlas.get_topk_rois(np.stack([full[n] for n in FN]).mean(0).mean(0), k=10))

out = dict(verdict=verdict, gates=dict(G1=bool(G1), G2=bool(G2), G3=bool(G3), G4=bool(G4)),
           face_vs_nonface={r['name']: {k: r[k] for k in ('d', 'U', 'p')} for r in [FFC, V1, EB, A1, FFCv]},
           confirm_PPA_scene_gt_face={k: PP[k] for k in ('d', 'U', 'p')},
           n_face=len(FN), n_nonface=len(NN), source='Charade (1963), public domain')
Path('/kaggle/working/gate0v2_results.json').write_text(json.dumps(out, indent=2))
print('\nwrote gate0v2_results.json + gate0v2_contrast.png to /kaggle/working - DOWNLOAD before the session ends')